In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd
import geopandas as gpd
import os
from pathlib import Path


In [2]:
cwd=os.getcwd()
curr_dir=Path(cwd).parent
data_dir=os.path.join(curr_dir,"data")
df = pd.read_csv(os.path.join(data_dir, "train(1).csv"))


In [3]:
df_test=pd.read_csv(os.path.join(data_dir,"test2(test(1)).csv"))
df_test

,id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2591820310,20141006T000000,4,2.25,2070,8893,2.0,0,0,4,8,2070,0,1986,0,98058,47.4388,-122.162,2390,7700
1,7974200820,20140821T000000,5,3.00,2900,6730,1.0,0,0,5,8,1830,1070,1977,0,98115,47.6784,-122.285,2370,6283
2,7701450110,20140815T000000,4,2.50,3770,10893,2.0,0,2,3,11,3770,0,1997,0,98006,47.5646,-122.129,3710,9685
3,9522300010,20150331T000000,3,3.50,4560,14608,2.0,0,2,3,12,4560,0,1990,0,98034,47.6995,-122.228,4050,14226
4,9510861140,20140714T000000,3,2.50,2550,5376,2.0,0,0,3,9,2550,0,2004,0,98052,47.6647,-122.083,2250,4050
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5399,7732500270,20140925T000000,4,2.50,2820,15000,2.0,0,0,4,9,2820,0,1985,0,98052,47.7255,-122.101,2440,15000
5400,3856903515,20141222T000000,3,2.00,1460,6250,1.5,0,0,4,7,1460,0,1912,0,98103,47.6693,-122.333,1690,4750
5401,2557000400,20150409T000000,3,2.50,2070,9900,1.0,0,0,3,8,1420,650,1979,0,98023,47.2988,-122.371,2070,8250
5402,4386700135,20141114T000000,4,2.25,4760,8036,2.5,0,0,5,9,3390,1370,1916,0,98112,47.6415,-122.285,2950,9323


### importing libraries from sentinelhub to download images

In [4]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt
import os 
from concurrent.futures import ThreadPoolExecutor

### using our client_id and client_secret in ap.txt

In [6]:
import os 

with open(os.path.join(curr_dir,"api.txt"),"r") as file:
    line=file.readlines()
    client_id = line[0].replace("Client ID:", "").strip()
    client_secret = line[1].replace("Client Secret:", "").strip()



In [7]:
from sentinelhub import SHConfig

config = SHConfig()
config.sh_client_id = client_id
config.sh_client_secret = client_secret
config.sh_base_url = "https://services.sentinel-hub.com"
config.save()
# Optional but recommended
config.instance_id = None
if not config.sh_client_id or not config.sh_client_secret:
    print("Warning! To use Process API, please provide the credentials (OAuth client ID and client secret).")
else:
    print("Credentials match and it is working")

Credentials match and it is working


In [8]:
import datetime
import os

import matplotlib.pyplot as plt
import numpy as np

from sentinelhub import (
    CRS,
    BBox,
    DataCollection,
    DownloadRequest,
    MimeType,
    MosaickingOrder,
    SentinelHubDownloadClient,
    SentinelHubRequest,
    bbox_to_dimensions,
)



In [11]:
from typing import Any
def plot_image(
    image: np.ndarray, factor: float = 1.0, clip_range: tuple[float, float] | None = None, **kwargs: Any
) -> None:
    """Utility function for plotting RGB images."""
    _, ax = plt.subplots(nrows=1, ncols=1, figsize=(15, 15))
    if clip_range is not None:
        ax.imshow(np.clip(image * factor, *clip_range), **kwargs)
    else:
        ax.imshow(image * factor, **kwargs)
    ax.set_xticks([])
    ax.set_yticks([])

### Defining our boundary box to be of 1 km

In [12]:
import math
# definig bounding box for image generation
def bbox_from_latlong(lat,long,size=560):
    lat_deg = size / 111320
    long_deg = size / (111320 * math.cos(math.radians(lat)))
    return BBox(
        bbox=[
            long-long_deg ,
            lat-lat_deg,
            long+long_deg,
            lat+lat_deg,
        ],
        crs=CRS.WGS84
    )


### wriiting evalscript and function to fetch satellite images

In [17]:
# writing the function to generate the images 
evalscript_true_color = """                                          
    //VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["B02", "B03", "B04"]
            }],
            output: {
                bands: 3
            }
        };
    }

    function evaluatePixel(sample) {
  if (sample.CLM == 1) {
    return [0.75 + sample.B04, sample.B03, sample.B02]
  }
  return [3.5*sample.B04, 3.5*sample.B03, 3.5*sample.B02];
}
"""


def fetch_satellite_image(lat,long,save_path):
    bbox=bbox_from_latlong(lat,long)
    size=bbox_to_dimensions(bbox,resolution=10)
    request=SentinelHubRequest(
        data_folder=".",
        evalscript=evalscript_true_color,
        input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2024-06-01", "2024-9-30"),
            mosaicking_order="leastCC"
        )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.PNG)
        ],
        bbox=bbox,
        size=size,
        config=config
    )
    img = request.get_data()[0]
    import cv2
    cv2.imwrite(save_path, img[:, :, ::-1])

### fetching images and storing in our folder

In [ ]:
start=0
os.makedirs("images", exist_ok=True)
def download_one(idx):
    row = df.iloc[idx]
    path = f"images/property_{idx}.png"

    if os.path.exists(path):
        return


    fetch_satellite_image(row.lat, row.long, path)
indices = range(start, len(df))
with ThreadPoolExecutor(max_workers=6) as executor:
    executor.map(download_one, indices)


### fetching test images and storing in our folder

In [18]:
start=0
os.makedirs("test_images", exist_ok=True)
def download_one(idx):
    row = df_test.iloc[idx]
    path = f"test_images/property_{idx}.png"
    if os.path.exists(path):
        return
    fetch_satellite_image(row.lat, row.long, path)
indices = range(start, len(df_test))
with ThreadPoolExecutor(max_workers=6) as executor:
    executor.map(download_one, indices)
